# A-Priori Algorithm

## Learning Objectives

By the end of this notebook you will be able to:

1. **Define** support, confidence, and interest for association rules
2. **State** the downward closure (anti-monotonicity) property and prove it
3. **Derive** the A-Priori candidate generation and pruning strategy
4. **Implement** multi-pass A-Priori for itemsets of arbitrary size
5. **Generate** association rules from frequent itemsets with confidence filtering


## Problem Statement

### Market Basket Model

A *basket* is a set of items purchased together in one transaction. The **frequent itemset mining** problem asks: which sets of items appear together in many baskets?

**Formally:**
- Universe of items $I = \{i_1, i_2, \ldots, i_m\}$
- $n$ baskets $B_1, B_2, \ldots, B_n$, each $B_k \subseteq I$
- **Support** of itemset $S$: $\text{sup}(S) = |\{k : S \subseteq B_k\}|$
- Itemset $S$ is *frequent* if $\text{sup}(S) \geq s$ for threshold $s$

### Association Rules

An *association rule* $A \Rightarrow B$ (where $A \cap B = \emptyset$) has:

$$\text{confidence}(A \Rightarrow B) = \frac{\text{sup}(A \cup B)}{\text{sup}(A)}$$

$$\text{interest}(A \Rightarrow B) = \text{confidence}(A \Rightarrow B) - P(B)$$

Interest filters out rules that hold only because $B$ is already very common.

### The Challenge

For $m$ items, there are $2^m - 1$ non-empty itemsets. For $m = 1000$, naïve enumeration is impossible. A-Priori avoids this with *downward closure*.


In [ ]:
from IPython.display import SVG, display

svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="400" font-family="monospace" font-size="12">
  <rect width="820" height="400" fill="#fafafa" rx="8"/>
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto">
      <polygon points="0 0,8 3,0 6" fill="#999"/>
    </marker>
  </defs>

  <!-- ═══ PASS 1: Count singletons ═══ -->
  <text x="90" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Pass 1: Count Singletons</text>

  <!-- Baskets -->
  <rect x="10" y="35" width="100" height="55" rx="4" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <text x="60" y="52" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Basket 1</text>
  <text x="60" y="68" text-anchor="middle" fill="#333" font-size="11">A B C</text>
  <text x="60" y="83" text-anchor="middle" fill="#333" font-size="11">s=2</text>

  <rect x="10" y="100" width="100" height="55" rx="4" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <text x="60" y="117" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Basket 2</text>
  <text x="60" y="133" text-anchor="middle" fill="#333" font-size="11">A C D</text>
  <text x="60" y="148" text-anchor="middle" fill="#333" font-size="11">s=2</text>

  <rect x="10" y="165" width="100" height="55" rx="4" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <text x="60" y="182" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Basket 3</text>
  <text x="60" y="198" text-anchor="middle" fill="#333" font-size="11">B C D</text>
  <text x="60" y="213" text-anchor="middle" fill="#333" font-size="11">s=2</text>

  <!-- Count table -->
  <rect x="130" y="35" width="90" height="120" rx="4" fill="white" stroke="#ccc" stroke-width="1"/>
  <text x="175" y="53" text-anchor="middle" fill="#555" font-size="11" font-weight="bold">item  count</text>
  <line x1="130" y1="58" x2="220" y2="58" stroke="#ccc" stroke-width="0.5"/>
  <text x="150" y="74" fill="#555" font-size="11">A</text><text x="205" y="74" text-anchor="middle" fill="#333" font-size="11">2</text>
  <text x="150" y="92" fill="#555" font-size="11">B</text><text x="205" y="92" text-anchor="middle" fill="#333" font-size="11">2</text>
  <text x="150" y="110" fill="#555" font-size="11">C</text><text x="205" y="110" text-anchor="middle" fill="#59a14f" font-size="11">3 ✓</text>
  <text x="150" y="128" fill="#555" font-size="11">D</text><text x="205" y="128" text-anchor="middle" fill="#333" font-size="11">2</text>
  <text x="175" y="148" text-anchor="middle" fill="#555" font-size="10">threshold s=2</text>

  <!-- ═══ CANDIDATE GEN: C2 ═══ -->
  <text x="370" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Candidate Gen (C₂)</text>

  <line x1="220" y1="95" x2="280" y2="95" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <rect x="285" y="35" width="175" height="215" rx="4" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="372" y="55" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">Frequent 1-itemsets → C₂</text>
  <text x="372" y="73" text-anchor="middle" fill="#555" font-size="10">{A},{B},{C},{D} → all pairs</text>
  <line x1="285" y1="80" x2="460" y2="80" stroke="#f28e2b" stroke-width="0.5" stroke-dasharray="3,3"/>
  <text x="330" y="98"  fill="#555" font-size="11">{A,B}</text>
  <text x="330" y="116" fill="#555" font-size="11">{A,C}</text>
  <text x="330" y="134" fill="#555" font-size="11">{A,D}</text>
  <text x="330" y="152" fill="#555" font-size="11">{B,C}</text>
  <text x="330" y="170" fill="#555" font-size="11">{B,D}</text>
  <text x="330" y="188" fill="#555" font-size="11">{C,D}</text>
  <text x="360" y="215" text-anchor="middle" fill="#555" font-size="10">Downward closure:</text>
  <text x="360" y="230" text-anchor="middle" fill="#555" font-size="10">all subsets are frequent</text>
  <text x="360" y="245" text-anchor="middle" fill="#555" font-size="10">→ valid candidates</text>

  <!-- ═══ PASS 2: Count pairs ═══ -->
  <text x="590" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Pass 2: Count Pairs → L₂</text>

  <line x1="460" y1="120" x2="510" y2="120" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <rect x="515" y="35" width="155" height="215" rx="4" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="592" y="55" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">Pair counts</text>
  <line x1="515" y1="60" x2="670" y2="60" stroke="#59a14f" stroke-width="0.5" stroke-dasharray="3,3"/>
  <text x="540" y="78"  fill="#555" font-size="11">{A,B}</text><text x="650" y="78"  text-anchor="middle" fill="#ccc" font-size="11">1 ✗</text>
  <text x="540" y="96"  fill="#555" font-size="11">{A,C}</text><text x="650" y="96"  text-anchor="middle" fill="#59a14f" font-size="11">2 ✓</text>
  <text x="540" y="114" fill="#555" font-size="11">{A,D}</text><text x="650" y="114" text-anchor="middle" fill="#59a14f" font-size="11">2 ✓</text>
  <text x="540" y="132" fill="#555" font-size="11">{B,C}</text><text x="650" y="132" text-anchor="middle" fill="#59a14f" font-size="11">2 ✓</text>
  <text x="540" y="150" fill="#555" font-size="11">{B,D}</text><text x="650" y="150" text-anchor="middle" fill="#ccc" font-size="11">1 ✗</text>
  <text x="540" y="168" fill="#555" font-size="11">{C,D}</text><text x="650" y="168" text-anchor="middle" fill="#59a14f" font-size="11">2 ✓</text>
  <text x="592" y="215" text-anchor="middle" fill="#555" font-size="10">L₂ = {{A,C},{A,D},{B,C},{C,D}}</text>
  <text x="592" y="232" text-anchor="middle" fill="#555" font-size="10">threshold s=2</text>

  <!-- ═══ Key theorem ═══ -->
  <rect x="10" y="295" width="790" height="90" rx="6" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="405" y="316" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">Downward Closure (Anti-Monotonicity)</text>
  <text x="405" y="338" text-anchor="middle" fill="#333" font-size="12">If itemset I is frequent, then every subset S ⊆ I is also frequent</text>
  <text x="405" y="358" text-anchor="middle" fill="#555" font-size="11">Contrapositive: if any subset S is infrequent → I is infrequent → prune I from candidates</text>
  <text x="405" y="376" text-anchor="middle" fill="#555" font-size="11">A-Priori exploits this to avoid counting exponentially many candidate itemsets</text>
</svg>
'''

display(SVG(svg))


## Downward Closure Property

**Theorem (Anti-Monotonicity):** If itemset $S$ is frequent (support $\geq s$), then every subset $T \subseteq S$ is also frequent.

**Proof:** Every basket containing $S$ also contains $T$ (since $T \subseteq S$). Therefore $\text{sup}(T) \geq \text{sup}(S) \geq s$. $\square$

**Contrapositive (used for pruning):** If $T$ is *infrequent*, then *every* superset of $T$ is also infrequent.

This means: to be a viable candidate for a frequent $k$-itemset, **all $(k-1)$-subsets must be frequent**. Any candidate failing this test can be pruned without counting it against the baskets.


## Derivation of the A-Priori Algorithm

### Pass 1 — Count Singletons

Read baskets once; maintain a count table $C_1[i]$ for each item $i$.

After the pass: $L_1 = \{\{i\} : C_1[i] \geq s\}$ — all frequent 1-itemsets.

**Memory:** $O(m)$ for item counts — fits in RAM even for $m = 10^6$.

### Candidate Generation for Pass $k$

Given $L_{k-1}$ (frequent $(k-1)$-itemsets), generate $C_k$ (candidate $k$-itemsets):

1. For each pair of $(k-1)$-itemsets $P$, $Q \in L_{k-1}$ that share exactly $k-2$ items:
   - Form $P \cup Q$ (a $k$-itemset)
   - **Prune:** check if all $(k-1)$-subsets of $P \cup Q$ are in $L_{k-1}$; if any is absent, discard

This guarantees $C_k \supseteq L_k$ (no false negatives) while $|C_k| \ll 2^m$ (no unnecessary candidates).

### Pass $k$ — Count Candidates

Read baskets again; for each basket $B$, check which candidates $c \in C_k$ satisfy $c \subseteq B$.

After the pass: $L_k = \{c \in C_k : \text{count}[c] \geq s\}$.

### Termination

Stop when $L_k = \emptyset$. Total passes = length of longest frequent itemset.

### Memory Requirement

At each pass, only $C_k$ and $L_k$ need to reside in RAM. If $|C_2|$ does not fit in memory, use PCY (see PCY notebook).


## Algorithm Steps

### Inputs

- Baskets $B_1, \ldots, B_n$ (on disk)
- Support threshold $s$

### Steps

1. **Pass 1:** Count all items; output $L_1 = \{$ items with count $\geq s \}$

2. **For $k = 2, 3, \ldots$:**

   a. **Generate** $C_k$ from $L_{k-1}$ using join + prune

   b. **Pass $k$:** Read baskets; for each basket count all $c \in C_k$ it contains

   c. **Output** $L_k = \{c \in C_k :$ count $\geq s\}$

   d. **Stop** if $L_k = \emptyset$

3. **Output** $\bigcup_k L_k$ — all frequent itemsets

4. *(Optional)* **Generate association rules** from frequent itemsets using confidence filtering

### Complexity

- **Passes:** at most $\ell + 1$ where $\ell$ = length of longest frequent itemset
- **Per pass:** $O(n \cdot \overline{|B|} \cdot |C_k|)$ where $\overline{|B|}$ = mean basket size
- **Memory:** $O(|C_k|)$ — dominated by candidate pairs in pass 2


In [ ]:
from collections import defaultdict
from itertools import combinations


def apriori(baskets, min_support):
    """
    A-Priori algorithm for mining frequent itemsets.

    Inputs
    ------
    baskets     : list of sets — each set is one transaction/basket
    min_support : int — minimum number of baskets an itemset must appear in

    Output
    ------
    frequent : dict {frozenset: count} — all frequent itemsets and their support counts
    """
    frequent = {}
    n_baskets = len(baskets)

    # ── Pass 1: count singletons ──────────────────────────────────────────────
    singleton_count = defaultdict(int)
    for basket in baskets:
        for item in basket:
            singleton_count[frozenset([item])] += 1

    L_prev = {fs: cnt for fs, cnt in singleton_count.items()
              if cnt >= min_support}
    frequent.update(L_prev)

    k = 2  # current itemset size
    while L_prev:
        # ── Candidate generation: join L_{k-1} pairs ─────────────────────────
        # Two (k-1)-itemsets merge into a candidate k-itemset if they share k-2 items
        prev_list = list(L_prev.keys())
        candidates = set()
        for i in range(len(prev_list)):
            for j in range(i + 1, len(prev_list)):
                union = prev_list[i] | prev_list[j]
                if len(union) == k:
                    # Downward closure check: every (k-1)-subset must be frequent
                    if all(frozenset(sub) in L_prev
                           for sub in combinations(union, k - 1)):
                        candidates.add(union)

        if not candidates:
            break

        # ── Count candidates over baskets ─────────────────────────────────────
        candidate_count = defaultdict(int)
        for basket in baskets:
            for cand in candidates:
                if cand <= basket:
                    candidate_count[cand] += 1

        L_k = {fs: cnt for fs, cnt in candidate_count.items()
               if cnt >= min_support}
        frequent.update(L_k)
        L_prev = L_k
        k += 1

    return frequent


def generate_association_rules(frequent, min_confidence):
    """
    Generate association rules from frequent itemsets.

    Inputs
    ------
    frequent       : dict {frozenset: count} from apriori()
    min_confidence : float — minimum confidence threshold

    Output
    ------
    rules : list of (antecedent, consequent, confidence, support)
    """
    rules = []
    for itemset, support in frequent.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for antecedent in map(frozenset, combinations(itemset, size)):
                consequent = itemset - antecedent
                if antecedent in frequent:
                    confidence = support / frequent[antecedent]
                    if confidence >= min_confidence:
                        rules.append((antecedent, consequent, confidence, support))

    return sorted(rules, key=lambda x: -x[2])


# ── Demo ──────────────────────────────────────────────────────────────────────
baskets = [
    {"bread", "milk", "eggs"},
    {"bread", "butter", "eggs"},
    {"milk", "butter", "cheese"},
    {"bread", "milk", "butter"},
    {"milk", "eggs", "cheese"},
    {"bread", "milk", "butter", "eggs"},
]

frequent = apriori(baskets, min_support=3)

print(f"Frequent itemsets (support ≥ 3):")
for itemset, count in sorted(frequent.items(), key=lambda x: (len(x[0]), -x[1])):
    items = ", ".join(sorted(itemset))
    print(f"  {{{items}}:<35} support = {count}")

print(f"\nAssociation rules (confidence ≥ 0.6):")
rules = generate_association_rules(frequent, min_confidence=0.6)
for ant, cons, conf, sup in rules[:10]:
    a = ", ".join(sorted(ant))
    c = ", ".join(sorted(cons))
    print(f"  {{{a}}} -> {{{c}:<20}}  conf={conf:.2f}  sup={sup}")
